In [1]:
# Install required libraries
!pip install pyspark
!pip install datasets
!pip install pyarrow
!pip install scikit-learn


In [ ]:
#Data Ingestion and Storage Design
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BrewerySalesDataEngineering") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

spark


In [ ]:
#Load Dataset from HuggingFace
from datasets import load_dataset

ds = load_dataset("Mathi65xl/Brewery_sales")

# Convert to Pandas first
import pandas as pd

df_pandas = ds['train'].to_pandas()

df_pandas.head()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Repo card metadata block was not found. Setting CardData to empty.


,Batch_ID,Brew_Date,Beer_Style,SKU,Location,Fermentation_Time,Temperature,pH_Level,Gravity,Alcohol_Content,Bitterness,Color,Ingredient_Ratio,Volume_Produced,Total_Sales,Quality_Score,Brewhouse_Efficiency,Loss_During_Brewing,Loss_During_Fermentation,Loss_During_Bottling_Kegging
0,7870796,2020-01-01 00:00:19,Wheat Beer,Kegs,Whitefield,16,24.204251,5.289845,1.039504,5.370842,20,5,1:0.32:0.16,4666,2664.759345,8.577016,89.195882,4.104988,3.235485,4.663204
1,9810411,2020-01-01 00:00:31,Sour,Kegs,Whitefield,13,18.086763,5.275643,1.059819,5.096053,36,14,1:0.39:0.24,832,9758.801062,7.420541,72.480915,2.676528,4.246129,2.044358
2,2623342,2020-01-01 00:00:40,Wheat Beer,Kegs,Malleswaram,12,15.539333,4.778016,1.037476,4.824737,30,10,1:0.35:0.16,2115,11721.087016,8.451365,86.322144,3.299894,3.109440,3.033880
3,8114651,2020-01-01 00:01:37,Ale,Kegs,Rajajinagar,17,16.418489,5.345261,1.052431,5.509243,48,18,1:0.35:0.15,3173,12050.177463,9.671859,83.094940,2.136055,4.634254,1.489889
4,4579587,2020-01-01 00:01:43,Stout,Cans,Marathahalli,18,19.144908,4.861854,1.054296,5.133625,57,13,1:0.46:0.11,4449,5515.077465,7.895334,88.625833,4.491724,2.183389,2.990630


In [ ]:
from datasets import load_dataset

ds = load_dataset("Mathi65xl/Brewery_sales")

# Save to parquet chunks
ds['train'].to_parquet("brewery_raw.parquet")



Repo card metadata block was not found. Setting CardData to empty.


Creating parquet from Arrow format:   0%|          | 0/10000 [00:00<?, ?ba/s]

1912745055

In [ ]:
spark_df = spark.read.parquet("brewery_raw.parquet")

spark_df.printSchema()
spark_df.show(5)


root
 |-- Batch_ID: long (nullable = true)
 |-- Brew_Date: string (nullable = true)
 |-- Beer_Style: string (nullable = true)
 |-- SKU: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Fermentation_Time: long (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- pH_Level: double (nullable = true)
 |-- Gravity: double (nullable = true)
 |-- Alcohol_Content: double (nullable = true)
 |-- Bitterness: long (nullable = true)
 |-- Color: long (nullable = true)
 |-- Ingredient_Ratio: string (nullable = true)
 |-- Volume_Produced: long (nullable = true)
 |-- Total_Sales: double (nullable = true)
 |-- Quality_Score: double (nullable = true)
 |-- Brewhouse_Efficiency: double (nullable = true)
 |-- Loss_During_Brewing: double (nullable = true)
 |-- Loss_During_Fermentation: double (nullable = true)
 |-- Loss_During_Bottling_Kegging: double (nullable = true)

+--------+-------------------+----------+----+------------+-----------------+------------------+-----------

In [ ]:
# Install dependencies
!pip install pyspark datasets pyarrow

from pyspark.sql import SparkSession
from datasets import load_dataset

spark = SparkSession.builder \
    .appName("BrewerySales") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory","3g") \
    .config("spark.executor.memory","3g") \
    .getOrCreate()

# Load huggingface dataset
ds = load_dataset("Mathi65xl/Brewery_sales")

# Write to parquet
ds['train'].to_parquet("brewery_raw.parquet")

# Load into Spark
spark_df = spark.read.parquet("brewery_raw.parquet")

spark_df.printSchema()
spark_df.show(5)


Repo card metadata block was not found. Setting CardData to empty.


Creating parquet from Arrow format:   0%|          | 0/10000 [00:00<?, ?ba/s]

root
 |-- Batch_ID: long (nullable = true)
 |-- Brew_Date: string (nullable = true)
 |-- Beer_Style: string (nullable = true)
 |-- SKU: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Fermentation_Time: long (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- pH_Level: double (nullable = true)
 |-- Gravity: double (nullable = true)
 |-- Alcohol_Content: double (nullable = true)
 |-- Bitterness: long (nullable = true)
 |-- Color: long (nullable = true)
 |-- Ingredient_Ratio: string (nullable = true)
 |-- Volume_Produced: long (nullable = true)
 |-- Total_Sales: double (nullable = true)
 |-- Quality_Score: double (nullable = true)
 |-- Brewhouse_Efficiency: double (nullable = true)
 |-- Loss_During_Brewing: double (nullable = true)
 |-- Loss_During_Fermentation: double (nullable = true)
 |-- Loss_During_Bottling_Kegging: double (nullable = true)

+--------+-------------------+----------+----+------------+-----------------+------------------+-----------

# **Task-1**

# 1.a.	Data Ingestion and Storage Design

In [ ]:
#Data Validation at Ingestion
from pyspark.sql.functions import col, count, when

# Check null values
null_counts = spark_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in spark_df.columns
])

null_counts.show()


+--------+---------+----------+---+--------+-----------------+-----------+--------+-------+---------------+----------+-----+----------------+---------------+-----------+-------------+--------------------+-------------------+------------------------+----------------------------+
|Batch_ID|Brew_Date|Beer_Style|SKU|Location|Fermentation_Time|Temperature|pH_Level|Gravity|Alcohol_Content|Bitterness|Color|Ingredient_Ratio|Volume_Produced|Total_Sales|Quality_Score|Brewhouse_Efficiency|Loss_During_Brewing|Loss_During_Fermentation|Loss_During_Bottling_Kegging|
+--------+---------+----------+---+--------+-----------------+-----------+--------+-------+---------------+----------+-----+----------------+---------------+-----------+-------------+--------------------+-------------------+------------------------+----------------------------+
|       0|        0|         0|  0|       0|                0|          0|       0|      0|              0|         0|    0|               0|              0|      

In [ ]:
from pyspark.sql import SparkSession
from datasets import load_dataset
from pyspark.sql.functions import year, col

# Initialize SparkSession if not already active
spark = SparkSession.builder \
    .appName("BrewerySales") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory","3g") \
    .config("spark.executor.memory","3g") \
    .getOrCreate()

# Load huggingface dataset if spark_df is not defined
ds = load_dataset("Mathi65xl/Brewery_sales")
ds['train'].to_parquet("brewery_raw.parquet")
spark_df = spark.read.parquet("brewery_raw.parquet")

# Create 'Year' and 'Region' columns
spark_df_partitioned = spark_df.withColumn("Year", year(col("Brew_Date")))
spark_df_partitioned = spark_df_partitioned.withColumn("Region", col("Location"))

# Now partition the DataFrame
spark_df_partitioned.write \
    .partitionBy("Year", "Region") \
    .mode("overwrite") \
    .parquet("/content/brewery_parquet")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Repo card metadata block was not found. Setting CardData to empty.


Creating parquet from Arrow format:   0%|          | 0/10000 [00:00<?, ?ba/s]

In [ ]:
#Read from Parquet
optimized_df = spark.read.parquet("/content/brewery_parquet")

optimized_df.show(5)


+--------+-------------------+----------+-------+-----------+-----------------+------------------+-----------------+------------------+------------------+----------+-----+----------------+---------------+------------------+-----------------+--------------------+-------------------+------------------------+----------------------------+----+-----------+
|Batch_ID|          Brew_Date|Beer_Style|    SKU|   Location|Fermentation_Time|       Temperature|         pH_Level|           Gravity|   Alcohol_Content|Bitterness|Color|Ingredient_Ratio|Volume_Produced|       Total_Sales|    Quality_Score|Brewhouse_Efficiency|Loss_During_Brewing|Loss_During_Fermentation|Loss_During_Bottling_Kegging|Year|     Region|
+--------+-------------------+----------+-------+-----------+-----------------+------------------+-----------------+------------------+------------------+----------+-----+----------------+---------------+------------------+-----------------+--------------------+-------------------+----------

# 1(b) Distributed Data Processing Pipeline

In [ ]:
#Broadcast Join Example
from pyspark.sql.functions import broadcast

# Example: assume region_lookup is small table
region_lookup = optimized_df.select("Region").distinct()

joined_df = optimized_df.join(
    broadcast(region_lookup),
    on="Region",
    how="inner"
)

joined_df.show(5)


+-----------+--------+-------------------+----------+-------+-----------+-----------------+------------------+-----------------+------------------+------------------+----------+-----+----------------+---------------+------------------+-----------------+--------------------+-------------------+------------------------+----------------------------+----+
|     Region|Batch_ID|          Brew_Date|Beer_Style|    SKU|   Location|Fermentation_Time|       Temperature|         pH_Level|           Gravity|   Alcohol_Content|Bitterness|Color|Ingredient_Ratio|Volume_Produced|       Total_Sales|    Quality_Score|Brewhouse_Efficiency|Loss_During_Brewing|Loss_During_Fermentation|Loss_During_Bottling_Kegging|Year|
+-----------+--------+-------------------+----------+-------+-----------+-----------------+------------------+-----------------+------------------+------------------+----------+-----+----------------+---------------+------------------+-----------------+--------------------+------------------

# Memory Management

In [ ]:
from pyspark import StorageLevel

optimized_df.persist(StorageLevel.MEMORY_AND_DISK)

optimized_df.count()  # Force evaluation

optimized_df.unpersist()


DataFrame[Batch_ID: bigint, Brew_Date: string, Beer_Style: string, SKU: string, Location: string, Fermentation_Time: bigint, Temperature: double, pH_Level: double, Gravity: double, Alcohol_Content: double, Bitterness: bigint, Color: bigint, Ingredient_Ratio: string, Volume_Produced: bigint, Total_Sales: double, Quality_Score: double, Brewhouse_Efficiency: double, Loss_During_Brewing: double, Loss_During_Fermentation: double, Loss_During_Bottling_Kegging: double, Year: int, Region: string]

In [ ]:
#Error Handling
try:
    transformed_df = optimized_df.withColumn("Total_Sales", col("Price") * col("Quantity"))
    transformed_df.show(5)
except Exception as e:
    print("Error occurred:", e)


{"ts": "2026-02-19 10:56:19.742", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Price` cannot be resolved. Did you mean one of the following? [`SKU`, `Year`, `Color`, `Gravity`, `Region`]. SQLSTATE: 42703", "context": {"file": "line 3 in cell [6]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o65.withColumn.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Price` cannot be resolved. Did you mean one of the following? [`SKU`, `Year`, `Color`, `Gravity`, `Region`]. SQLSTATE: 42703;\n'Project [Batch_ID#24L, Brew_Date#25, Beer_Style#26, SKU#27, Location#28, Fermentation_Time#29L, Temperature#30, pH_Level#31, Gravity#32, Alcohol_Content#33, Bitterness#34L, Color#35L, Ingredient_Ratio#3

Error occurred: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Price` cannot be resolved. Did you mean one of the following? [`SKU`, `Year`, `Color`, `Gravity`, `Region`]. SQLSTATE: 42703;
'Project [Batch_ID#24L, Brew_Date#25, Beer_Style#26, SKU#27, Location#28, Fermentation_Time#29L, Temperature#30, pH_Level#31, Gravity#32, Alcohol_Content#33, Bitterness#34L, Color#35L, Ingredient_Ratio#36, Volume_Produced#37L, '`*`('Price, 'Quantity) AS Total_Sales#890, Quality_Score#39, Brewhouse_Efficiency#40, Loss_During_Brewing#41, Loss_During_Fermentation#42, Loss_During_Bottling_Kegging#43, Year#44, Region#45]
+- Relation [Batch_ID#24L,Brew_Date#25,Beer_Style#26,SKU#27,Location#28,Fermentation_Time#29L,Temperature#30,pH_Level#31,Gravity#32,Alcohol_Content#33,Bitterness#34L,Color#35L,Ingredient_Ratio#36,Volume_Produced#37L,Total_Sales#38,Quality_Score#39,Brewhouse_Efficiency#40,Loss_During_Brewing#41,Loss_During_Fermentation#42,Loss_During_Bottling_Keggi

# 1(c) Performance Optimization

In [ ]:
#DataFrame vs RDD Justification
optimized_df.explain(True)


== Parsed Logical Plan ==
UnresolvedDataSource format: parquet, isStreaming: false, paths: 1 provided

== Analyzed Logical Plan ==
Batch_ID: bigint, Brew_Date: string, Beer_Style: string, SKU: string, Location: string, Fermentation_Time: bigint, Temperature: double, pH_Level: double, Gravity: double, Alcohol_Content: double, Bitterness: bigint, Color: bigint, Ingredient_Ratio: string, Volume_Produced: bigint, Total_Sales: double, Quality_Score: double, Brewhouse_Efficiency: double, Loss_During_Brewing: double, Loss_During_Fermentation: double, Loss_During_Bottling_Kegging: double, Year: int, Region: string
Relation [Batch_ID#24L,Brew_Date#25,Beer_Style#26,SKU#27,Location#28,Fermentation_Time#29L,Temperature#30,pH_Level#31,Gravity#32,Alcohol_Content#33,Bitterness#34L,Color#35L,Ingredient_Ratio#36,Volume_Produced#37L,Total_Sales#38,Quality_Score#39,Brewhouse_Efficiency#40,Loss_During_Brewing#41,Loss_During_Fermentation#42,Loss_During_Bottling_Kegging#43,Year#44,Region#45] parquet

== Opt

In [ ]:
#Caching Strategy
optimized_df.cache()
optimized_df.count()


10000000

In [ ]:
#Shuffle Management
spark.conf.set("spark.sql.shuffle.partitions", "100")


In [ ]:
optimized_df = optimized_df.repartition(100, "Region")


# **Task-2 Scalability & Distributed ML**

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["Total_Sales", "Volume_Produced"],
    outputCol="features"
)

ml_df = assembler.transform(optimized_df)

In [ ]:
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)


# ML Algorithm 1: Linear Regression

In [ ]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="Total_Sales"
)

lr_model = lr.fit(train_df)

lr_predictions = lr_model.transform(test_df)


# ML Algorithm 2: Random Forest

In [ ]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="Total_Sales",
    numTrees=50
)

rf_model = rf.fit(train_df)
rf_predictions = rf_model.transform(test_df)


# ML Algorithm 3: Gradient Boosted Trees

In [ ]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="Total_Sales"
)

gbt_model = gbt.fit(train_df)


# Compare with scikit-learn Baseline

In [ ]:
from sklearn.linear_model import LinearRegression as SkLR

X_train = train_df.select("Total_Sales", "Volume_Produced").toPandas()
y_train = train_df.select("Total_Sales").toPandas()

sk_model = SkLR()
sk_model.fit(X_train, y_train)

LinearRegression()

In [ ]:
lr_model.save("/content/lr_model")

import pickle
pickle.dump(sk_model, open("/content/sk_model.pkl", "wb"))


# 2(b) CrossValidator

In [15]:
from pyspark.ml.regression import RandomForestRegressor

configs = [
    (10, 3),
    (20, 5),
    (50, 5)
]

for trees, depth in configs:
    print(f"\nTraining model: trees={trees}, depth={depth}")

    rf = RandomForestRegressor(
        numTrees=trees,
        maxDepth=depth,
        featuresCol="features",
        labelCol="Total_Sales"
    )

    model = rf.fit(train_df)
    preds = model.transform(test_df)

    rmse = evaluator.evaluate(preds)
    print("RMSE:", rmse)


Training model: trees=10, depth=3
RMSE: 1819.8631577539002

Training model: trees=20, depth=5
RMSE: 984.9533581627202

Training model: trees=50, depth=5
RMSE: 1027.3302791700812


# 2(c) Scalability Analysis

In [ ]:
#Strong Scaling Experiment
for partitions in [50, 100, 200]:
    df_test = optimized_df.repartition(partitions)
    print(partitions, df_test.count())


50 10000000
100 10000000
200 10000000


# **# Task-4 **
# Evaluation Metrics

In [7]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

# Ensure SparkSession is active and accessible
spark = SparkSession.builder.getOrCreate()

# Re-create ml_df, train_df, test_df, and lr_predictions if they are not defined
# (assuming 'optimized_df' is still available from previous cells)
if 'optimized_df' in locals():
    assembler = VectorAssembler(
        inputCols=["Total_Sales", "Volume_Produced"],
        outputCol="features"
    )
    ml_df = assembler.transform(optimized_df)
    train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

    lr = LinearRegression(
        featuresCol="features",
        labelCol="Total_Sales"
    )
    lr_model = lr.fit(train_df)
    lr_predictions = lr_model.transform(test_df)
else:
    print("Warning: optimized_df not found. Please ensure previous data loading and preprocessing cells are executed.")



evaluator = RegressionEvaluator(
    labelCol="Total_Sales",
    predictionCol="prediction",
    metricName="rmse"
)

# Check if lr_predictions is defined before evaluating
if 'lr_predictions' in locals() and lr_predictions is not None:
    rmse = evaluator.evaluate(lr_predictions)
    print("RMSE:", rmse)
else:
    print("Error: lr_predictions could not be generated. Please check previous steps.")

Error: lr_predictions could not be generated. Please check previous steps.


In [11]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.sql.functions import year, col
from datasets import load_dataset

# Ensure SparkSession is active and accessible
spark = SparkSession.builder.getOrCreate()

# Re-create ml_df, train_df, test_df, and lr_predictions if they are not defined
# (assuming 'optimized_df' is still available from previous cells)

# If optimized_df is not in scope, re-run the data ingestion and partitioning steps
if 'optimized_df' not in locals() or optimized_df is None:
    print("optimized_df not found. Re-running data loading and partitioning.")
    # Load huggingface dataset
    ds = load_dataset("Mathi65xl/Brewery_sales")
    # Write to parquet
    ds['train'].to_parquet("brewery_raw.parquet")
    # Load into Spark
    spark_df = spark.read.parquet("brewery_raw.parquet")

    # Create 'Year' and 'Region' columns
    spark_df_partitioned = spark_df.withColumn("Year", year(col("Brew_Date")))
    spark_df_partitioned = spark_df_partitioned.withColumn("Region", col("Location"))

    # Write to partitioned parquet
    spark_df_partitioned.write \
        .partitionBy("Year", "Region") \
        .mode("overwrite") \
        .parquet("/content/brewery_parquet")

    # Read from partitioned parquet to get optimized_df
    optimized_df = spark.read.parquet("/content/brewery_parquet")

# 1. Feature Engineering: Create a VectorAssembler
assembler = VectorAssembler(
    inputCols=["Total_Sales", "Volume_Produced"],
    outputCol="features"
)
ml_df = assembler.transform(optimized_df)

# 2. Split data into training and testing sets
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

# 3. ML Algorithm 1: Linear Regression
lr = LinearRegression(
    featuresCol="features",
    labelCol="Total_Sales"
)
lr_model = lr.fit(train_df)
lr_predictions = lr_model.transform(test_df)

# Evaluate the Linear Regression model
evaluator = RegressionEvaluator(
    labelCol="Total_Sales",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator.evaluate(lr_predictions)
print("RMSE:", rmse)


optimized_df not found. Re-running data loading and partitioning.


Repo card metadata block was not found. Setting CardData to empty.


Creating parquet from Arrow format:   0%|          | 0/10000 [00:00<?, ?ba/s]

RMSE: 3.6257506794332207e-12


In [13]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import lit


evaluator = RegressionEvaluator(
    labelCol="Total_Sales",
    predictionCol="prediction",
    metricName="rmse"
)

# Calculate RMSE for the existing lr_predictions as a base case or one bootstrap sample
rmse_result = evaluator.evaluate(lr_predictions)
print(f"RMSE from lr_predictions: {rmse_result}")

# Example of what a bootstrap analysis might start to look like (conceptual):
print("\nStarting a conceptual bootstrap analysis (showing only one re-evaluation here):")

# Simulate one bootstrap sample evaluation (re-using existing predictions for simplicity here)
# In a real scenario, you'd generate new models and predictions per sample
bootstrap_sample_rmse = evaluator.evaluate(lr_predictions.withColumn("Total_Sales", col("Total_Sales") + lit(0.01))) # Small perturbation for demonstration
print(f"Conceptual bootstrap sample RMSE (with small perturbation): {bootstrap_sample_rmse}")

# Real bootstrap analysis would collect many such RMSE values and analyze their distribution.

RMSE from lr_predictions: 3.6257506794332207e-12

Starting a conceptual bootstrap analysis (showing only one re-evaluation here):
Conceptual bootstrap sample RMSE (with small perturbation): 0.009999999998900383
